# MNIST CNN: Comparing PyTorch Adam/AdamW vs From-Scratch Implementations

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import einops
from typing import Iterator, Callable
import math

## Tiny CNN for MNIST

In [2]:
class TinyCNN(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.conv1: nn.Conv2d = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.conv2: nn.Conv2d = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool: nn.MaxPool2d = nn.MaxPool2d(2)
        self.fc: nn.Linear = nn.Linear(16 * 7 * 7, 10)
        self.relu: nn.ReLU = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 1, 28, 28)
        x = self.relu(self.conv1(x))   # (B, 8, 28, 28)
        x = self.pool(x)               # (B, 8, 14, 14)
        x = self.relu(self.conv2(x))   # (B, 16, 14, 14)
        x = self.pool(x)               # (B, 16, 7, 7)
        x = einops.rearrange(x, 'B C H W -> B (C H W)')
        x = self.fc(x)                 # (B, 10)
        return x

## Training Loop

In [3]:
def train(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    n_epochs: int = 3,
    batch_size: int = 256,
    device: str = "cpu",
) -> list[float]:
    transform: transforms.Compose = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
    train_data: datasets.MNIST = datasets.MNIST(
        root="./data", train=True, download=True, transform=transform
    )
    loader: DataLoader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    loss_fn: nn.CrossEntropyLoss = nn.CrossEntropyLoss()

    model.to(device)
    model.train()
    epoch_losses: list[float] = []

    for epoch in range(n_epochs):
        running_loss: float = 0.0
        n_batches: int = 0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits: torch.Tensor = model(images)
            loss: torch.Tensor = loss_fn(
                einops.rearrange(logits, 'B V -> B V'),
                labels,
            )
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            n_batches += 1
        avg_loss: float = running_loss / n_batches
        epoch_losses.append(avg_loss)
        print(f"  Epoch {epoch + 1}/{n_epochs} — loss: {avg_loss:.4f}")

    return epoch_losses

## Train with PyTorch Adam & AdamW

In [4]:
print("=== PyTorch Adam ===")
model_adam: TinyCNN = TinyCNN()
optimizer_adam: torch.optim.Adam = torch.optim.Adam(model_adam.parameters(), lr=1e-3)
losses_adam: list[float] = train(model_adam, optimizer_adam)

print("\n=== PyTorch AdamW ===")
model_adamw: TinyCNN = TinyCNN()
optimizer_adamw: torch.optim.AdamW = torch.optim.AdamW(model_adamw.parameters(), lr=1e-3, weight_decay=0.01)
losses_adamw: list[float] = train(model_adamw, optimizer_adamw)

=== PyTorch Adam ===


100.0%
100.0%
100.0%
100.0%


  Epoch 1/3 — loss: 0.4777
  Epoch 2/3 — loss: 0.1098
  Epoch 3/3 — loss: 0.0783

=== PyTorch AdamW ===
  Epoch 1/3 — loss: 0.4442
  Epoch 2/3 — loss: 0.1323
  Epoch 3/3 — loss: 0.0934


## From-Scratch Adam

In [ ]:
class ScratchAdam(torch.optim.Optimizer):
    def __init__(self, 
                 params: Iterator[nn.Parameter],
                 lr: float = 1e-3,
                 beta1: float = 0.9,
                 beta2: float = 0.99,
                 eps: float = 1e-8,
                 wd: float = 1e-2
    )->None:
        defaults: dict = dict(lr=lr,beta1 = beta1, beta2=beta2, eps=eps, wd=wd)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure: Callable | None = None)-> torch.Tensor | None:
        # initialize loss to None if we don't have a closure function to call first
        # else get loss from the closure function
        loss: torch.Tensor | None = None
        if closure is not None:
            # you obviously want gradients flowing if we need to calculate the loss via a closure function
            with torch.enable_grad():
                loss = closure()

        # for group in groups
        for group in self.param_groups:
            # for param in this gorup
            for parameter in group['params']:
                # skip if no grad
                if parameter.grad is None:
                    continue

                # initialize the momentum buffers
                state = self.state[parameter]
                if len(state) == 0:
                    state['step'] = 0
                    state['m1'] = torch.zeros_like(parameter)
                    state['m2'] = torch.zeros_like(parameter)

                # fetch current buffers
                state['step'] += 1
                step = state['step']
                m1 = state['m1']
                m2 = state['m2']

                # fetch current adam param values for group
                lr = group['lr']
                beta1 = group['beta1']
                beta2 = group['beta2']
                wd = group['wd']
                eps = group['eps']


                # what is the right way to add weight decay in this from-scratch adam implementaitons step() function (the weight decay constant for this param group is in wd)?
                parameter.grad = parameter.grad + wd * parameter

                # update buffer values (what are the formulas for updating the two moments?)
                m1 = beta1 * m1 + (1 - beta1)*parameter.grad
                m2 = beta2 * m2 + (1 - beta2)*(parameter.grad*parameter.grad)

                # debias (I don't really understand why this works. I also don't understand the mathematical sense in which these two buffers are moments)
                # modify this to use step in the correct way for adam
                m1_hat = m1 / (1 - beta1 ** step)
                m2_hat = m2 / (1 - beta2 ** step)

                # In Adam, do we write the debiased buffer or the original back to the state dict?)
                state['m1'] = m1
                state['m2'] = m2
                # what's wrong with this line for updating parameters in a from-scratch adam optimizer? 
                # this won't update the model weights because (doing paramater = parameter - ...) only modifies the local python variable,
                # parameter = parameter - lr * (m1_hat / (torch.sqrt(m2_hat) + eps))
                
                # to fix you must either update in-place 
                # parameter -= lr * (m1_hat / (torch.sqrt(m2_hat) + eps))

                # or directly replace the contents of the parameters data field (where the weights get stored) like this
                parameter.data = parameter.data - lr * (m1_hat / (torch.sqrt(m2_hat) + eps))
                
        return loss


        # pseudocode steps for Adam
        # for param in param groups
            # if param not in self.state
                # initialize momentum buffers
            # read out previous buffer values and hyperparam values from group state
            # shift gradient by weight decay term
            # calculate raw momentum buffer updates
            # calculate debiased momentum buffers
            # update state dict values for momentum buffers using (raw or debiasd) raw momentum buffers
            # update the gradients using teh (raw or debiased) debiased momentum buffers 



In [ ]:
class ScratchAdamW(torch.optim.Optimizer):
    def __init__(self, 
                 params: Iterator[nn.Parameter],
                 lr: float = 1e-3,
                 beta1: float = 0.9,
                 beta2: float = 0.99,
                 eps: float = 1e-8,
                 wd: float = 1e-2
    )->None:
        defaults: dict = dict(lr=lr,beta1 = beta1, beta2=beta2, eps=eps, wd=wd)
        super().__init__(params, defaults)
    
    @torch.no_grad()
    def step(self, closure: Callable | None = None)->torch.Tensor | None:
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        
        for group in self.param_groups:
            for parameter in group['params']:

                grad = parameter.grad
                if grad is None:
                    continue
                
                state = self.state[parameter]
                if len(state) == 0:
                    state['step'] = 0
                    state['m1'] = torch.zeros_like(grad)
                    state['m2'] = torch.zeros_like(grad)
                
                state['step'] += 1
                step = state['step']
                m1 = state['m1']
                m2 = state['m2']

                beta1 = group['beta1']
                beta2 = group['beta2']
                lr = group['lr']
                eps = group['eps']
                wd = group['wd']

                m1 = m1 * beta1 + (1 - beta1) * grad
                m2 = m2 * beta2 + (1 - beta2) * grad * grad


                state['m1'] = m1
                state['m2'] = m2

                m1_hat = m1 / (1 - beta1 ** step)
                m2_hat = m2 / (1 - beta2 ** step)

                parameter.data = parameter.data - (lr * m1_hat) / (torch.sqrt(m2_hat) + eps)

                # what is the right way of doing decoupled weight decay in this fromscratch implementation of AdamW? 
                parameter.data = parameter.data - (lr * wd * parameter.data)

        return loss


                


            # the primary difference between Adam and AdamW is that Adam does coupled weight decay (if you choose to do weight decay)
            # this means that the gradient is actually shifted by the weight decay term (g <- g + lambda*theta (weight decay)). AdamW does uncoupled weight decay. this means that it decays model weights without 
            # scaling the gradient accordingly (so it scales model weights independently of the gradient norms)
                


In [ ]:
# from scratch Muon

## Train with From-Scratch Adam & AdamW

In [ ]:
print("=== Scratch Adam (no weight decay)===")
model_scratch_adam: TinyCNN = TinyCNN()
optimizer_scratch_adam: ScratchAdam = ScratchAdam(model_scratch_adam.parameters(), lr=1e-3, wd=0.0)
losses_scratch_adam: list[float] = train(model_scratch_adam, optimizer_scratch_adam)
print("=== Scratch Adam (with weight decay)===")
model_scratch_adam: TinyCNN = TinyCNN()
optimizer_scratch_adam: ScratchAdam = ScratchAdam(model_scratch_adam.parameters(), lr=1e-3, wd=0.01)
losses_scratch_adam: list[float] = train(model_scratch_adam, optimizer_scratch_adam)
# print(optimizer_scratch_adam.state)
# print(optimizer_scratch_adam.param_groups)

=== Scratch Adam (no weight decay)===
  Epoch 1/3 — loss: 0.4630
  Epoch 2/3 — loss: 0.1161
  Epoch 3/3 — loss: 0.0805
=== Scratch Adam (with weight decay)===
  Epoch 1/3 — loss: 0.4833
  Epoch 2/3 — loss: 0.1373
  Epoch 3/3 — loss: 0.1010


In [25]:

print("\n=== Scratch AdamW ===")
model_scratch_adamw: TinyCNN = TinyCNN()
optimizer_scratch_adamw: ScratchAdamW = ScratchAdamW(model_scratch_adamw.parameters(), lr=1e-3, wd=0.01)
losses_scratch_adamw: list[float] = train(model_scratch_adamw, optimizer_scratch_adamw)


=== Scratch AdamW ===
  Epoch 1/3 — loss: 0.4677
  Epoch 2/3 — loss: 0.1422
  Epoch 3/3 — loss: 0.0917
